# Lightweight Adaptive NeSy-Gen — one-day Colab run

This deadline-oriented notebook replaces 3–4B parameter drafters with a frozen **DeiT-tiny** encoder and **DistilGPT-2** decoder (about 90M parameters total). It generates each test report once, then replays every PrimeKG/LTN/gate ablation. BLEU-1 ≥0.50 is a measured target, never a hard-coded or guaranteed result.

Use an A100 if available; L4/T4 also work with smaller batches. Start with IU-Xray, freeze decisions using validation data, and only then run MIMIC-CXR.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUN_DATASET = 'iuxray_official'  # later: 'mimic_aug'
GIT_REF = 'codex/lightweight-one-day'
REPO_URL = 'https://github.com/FaezehMillerAI/NesyGENAAAI.git'
REPO_DIR = '/content/NesyGENAAAI'
BLEU1_TARGET = 0.50
RUN_DRAFTING_BASELINES = True  # query-only neural + top-1 retrieval
RUN_HEAVY_PUBLICATION_EVAL = True
print({'dataset': RUN_DATASET, 'git_ref': GIT_REF, 'bleu1_target': BLEU1_TARGET})

In [ ]:
import os, pathlib, shutil, subprocess, sys
repo = pathlib.Path(REPO_DIR)
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', GIT_REF, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', GIT_REF], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', GIT_REF], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lightweight,dev]'], check=True)
print(subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
import json, torch
from pathlib import Path
paths = json.loads(Path('configs/colab_paths.json').read_text())[RUN_DATASET]
MANIFEST = Path(paths['manifest'])
MEDSIGLIP_CACHE = Path(paths['medsiglip_cache'])
PRIMEKG_CACHE = Path(paths['primekg_cache'])
OUTPUT_ROOT = Path('/content/drive/MyDrive/aaai_2026_experiments/adaptive_nesy_gen_lightweight') / RUN_DATASET
MODEL_ROOT = OUTPUT_ROOT / 'deit_tiny_distilgpt2'
MODEL_PATH = MODEL_ROOT / 'best_model'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for path in (MANIFEST, MEDSIGLIP_CACHE, PRIMEKG_CACHE):
    assert path.exists(), f'Missing required artifact: {path}'
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU'
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
BATCH_SIZE = 32 if gpu_gb >= 35 else (12 if gpu_gb >= 20 else 8)
GRAD_ACCUM = max(1, 32 // BATCH_SIZE)
MAX_STEPS = 1500 if RUN_DATASET == 'iuxray_official' else 2500
print({'gpu': torch.cuda.get_device_name(0), 'gpu_gb': round(gpu_gb,1), 'batch': BATCH_SIZE, 'accum': GRAD_ACCUM, 'steps': MAX_STEPS})

In [ ]:
# Fast local contracts before downloading model weights.
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)

## Train once (resumable)
The test reports are redacted at ingestion. Training uses train/validation only, freezes DeiT-tiny, and saves the lowest validation-loss checkpoint. Rerunning this cell resumes automatically.

In [ ]:
train_cmd = [
    sys.executable, 'scripts/train_lightweight_vlm.py',
    '--manifest', str(MANIFEST),
    '--medsiglip-cache', str(MEDSIGLIP_CACHE),
    '--output-dir', str(MODEL_ROOT),
    '--max-steps', str(MAX_STEPS),
    '--batch-size', str(BATCH_SIZE),
    '--gradient-accumulation', str(GRAD_ACCUM),
]
subprocess.run(train_cmd, check=True)
assert (MODEL_PATH / 'config.json').exists()
print((MODEL_ROOT / 'training_summary.json').read_text())

## Generate the official test split once
Generation never loads test references. The MedSigLIP index remains training-only. Accept the gated `google/medsiglip-448` terms and run `huggingface-cli login` if the cache/query encoder asks for access.

In [ ]:
RAW_DRAFTS = OUTPUT_ROOT / 'lightweight_few_shot_full_adaptive.jsonl'
if not RAW_DRAFTS.exists():
    subprocess.run([
        sys.executable, 'scripts/run_experiments.py',
        '--manifest', str(MANIFEST), '--medsiglip-cache', str(MEDSIGLIP_CACHE),
        '--primekg-cache', str(PRIMEKG_CACHE), '--backend', 'lightweight',
        '--model-path', str(MODEL_PATH), '--drafting-mode', 'few-shot',
        '--ablation', 'drafting_only', '--output', str(RAW_DRAFTS),
    ], check=True)
print('drafts:', RAW_DRAFTS, 'rows:', sum(1 for _ in RAW_DRAFTS.open()))

## Replay all verifier experiments
The model is not loaded here. Retrieval results are memoized, so the same image is encoded once—not once per ablation. Interrupted runs can simply rerun this cell.

In [ ]:
SUITE_DIR = OUTPUT_ROOT / 'replay_lightweight_few_shot_suite'
subprocess.run([
    sys.executable, 'scripts/run_experiments.py',
    '--manifest', str(MANIFEST), '--medsiglip-cache', str(MEDSIGLIP_CACHE),
    '--primekg-cache', str(PRIMEKG_CACHE), '--backend', 'replay',
    '--drafts-jsonl', str(RAW_DRAFTS), '--suite', '--output', str(SUITE_DIR),
], check=True)
if RUN_DRAFTING_BASELINES:
    baselines = [
        ('lightweight_zero_shot', ['--backend', 'lightweight', '--model-path', str(MODEL_PATH), '--drafting-mode', 'zero-shot']),
        ('retrieval_only', ['--backend', 'retrieval', '--drafting-mode', 'few-shot']),
    ]
    for name, backend_args in baselines:
        output = SUITE_DIR / f'{name}.jsonl'
        if output.exists():
            continue
        subprocess.run([
            sys.executable, 'scripts/run_experiments.py',
            '--manifest', str(MANIFEST), '--medsiglip-cache', str(MEDSIGLIP_CACHE),
            '--primekg-cache', str(PRIMEKG_CACHE), '--ablation', 'drafting_only',
            '--output', str(output), *backend_args,
        ], check=True)
print('completed experiment files:', len(list(SUITE_DIR.glob('*.jsonl'))))

## Publication evaluation and the BLEU-1 target
This is the first stage that loads test references. It rejects missing/duplicate predictions and writes official metrics, paired bootstrap intervals, integrity/leakage audits, and blinded review packets.

In [ ]:
if RUN_HEAVY_PUBLICATION_EVAL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[eval]'], check=True)
    EVAL_DIR = OUTPUT_ROOT / 'publication_evaluation'
    subprocess.run([
        sys.executable, 'scripts/evaluate_publication.py',
        '--manifest', str(MANIFEST), '--predictions-dir', str(SUITE_DIR),
        '--primekg-cache', str(PRIMEKG_CACHE), '--output-dir', str(EVAL_DIR),
        '--baseline', 'rag_without_graph_ltn', '--bootstrap-samples', '2000',
    ], check=True)
    metrics = json.loads((EVAL_DIR / 'publication_metrics.json').read_text())
    score = metrics['methods']['full_adaptive']['official_metrics']['bleu_1']
    print({'official_bleu_1': score, 'target': BLEU1_TARGET, 'target_reached': score >= BLEU1_TARGET})
else:
    print('Heavy evaluation skipped; set RUN_HEAVY_PUBLICATION_EVAL=True and rerun.')

## Deadline strategy
1. Finish IU-Xray end to end and inspect validation/test BLEU plus clinical metrics.
2. Do not tune on test references. Adjust steps, retrieval probability, beam count, or thresholds using validation only.
3. Once frozen, change `RUN_DATASET` to `mimic_aug`, restart from the top, and keep the same settings.
4. A BLEU-1 miss is a result, not a software failure. Report confidence intervals and clinical fidelity alongside lexical overlap.